# Private HASS SFT · remote operator
This notebook starts from a frozen dataset. Prepare one using the dataset sections of the HASS end-to-end notebook, then bring its manifest URI and SHA-256 here. Jupyter submits and watches jobs, reads receipts, and displays reports.
Copy `configs/training-workspace.env.example` to `$HOME/eliza/training/.env`, fill in the global settings once, and install the current lightweight training client. See the [operator guide](../docs/runs/hass-remote-operator.md).

In [ ]:
from pathlib import Path
import re
import sys
import yaml
from IPython.display import clear_output

WORKSPACE = Path.home() / 'eliza/training'
# Optional overlay when the image client is older; must precede client imports.
if (WORKSPACE / '.client/nlp_training').is_dir():
    sys.path.insert(0, str(WORKSPACE / '.client'))
from nlp_training.operator import workspace_settings, submit_prepared, read_report, format_status
from eliza_storage import ElizaStorage
from eliza_training import ElizaTraining

RUN_LABEL = 'hass-qwen08-01'
MODEL_PRESET = 'sft_hass_private_smoke.template.yaml'
MANIFEST_URI = ''
MANIFEST_SHA256 = ''
SOURCE_RUN_ID = ''  # set to job 8's run UUID to evaluate its archived adapter directly
RESOURCE_OVERRIDES = {}  # e.g. {'memory_mb': 49152}
SETTING_OVERRIDES = {}

if not re.fullmatch(r'[a-zA-Z0-9][a-zA-Z0-9_-]*', RUN_LABEL):
    raise ValueError('Enter a run label containing only letters, numbers, underscores or hyphens')
settings, origins = workspace_settings(WORKSPACE, SETTING_OVERRIDES)
REPO = Path(settings['NLP_LAB_REPO']).expanduser()
RUN = WORKSPACE / 'runs' / RUN_LABEL
storage = ElizaStorage(endpoint_url=settings.get('STORAGE_URL'))
credential_file = settings.get('CONTROL_DAEMON_TOKEN_FILE')
control_token = Path(credential_file).expanduser().read_text().strip() if credential_file else None
training = ElizaTraining(bucket=settings['SLURM_ARTIFACT_BUCKET'], storage=storage,
                         control_url=settings['INFERENCE_CONTROL_URL'], endpoint_url=settings.get('SLURM_REST_URL'),
                         control_token=control_token)
print({k: (settings[k], origins[k]) for k in settings if not k.endswith('_FILE')})
print('DGX mode:', training.environment().get('phase'))
pin = dict(uri=MANIFEST_URI, sha256=MANIFEST_SHA256)
match = re.fullmatch(r's3://([^/]+)/dataset_id=([^/]+)/dataset_version=([^/]+)/manifest\.json', MANIFEST_URI)
if not match or not re.fullmatch(r'[a-f0-9]{64}', MANIFEST_SHA256):
    raise ValueError('Enter the exact frozen manifest URI and byte SHA-256')
bucket, dataset_id, version = match.groups()
cfg = yaml.safe_load((REPO / 'configs' / MODEL_PRESET).read_text())
cfg['data'].update(bucket=bucket, dataset_id=dataset_id, dataset_version=version, expected_manifest_sha256=MANIFEST_SHA256)
cfg['tracking']['mlflow_tracking_uri'] = settings['MLFLOW_TRACKING_URI']
resources = {'cpus': 8, 'memory_mb': 32768, 'time_minutes': 90, **RESOURCE_OVERRIDES}
print('Model:', cfg['model']['name'], '| Dataset:', pin, '| Resources:', resources)

def watch(run_id):
    training.discover()
    def update(status):
        clear_output(wait=True)
        print(format_status(status), flush=True)
    result = training.watch(run_id, timeout_seconds=7200, on_update=update)
    if result.get('outcome') != 'succeeded':
        print('Inspect the private train.log and worker/scheduler receipts for', run_id)
        raise RuntimeError(format_status(result))
    return result

def launch(kind, config):
    return submit_prepared(training, REPO, RUN / kind, kind=kind, config=config, pin=pin,
                           resources=resources, settings=settings)


## Measure context remotely
Enter `SUBMIT` to switch the DGX to training mode and run the tokenizer check in an allocation. It reports the native limit and measured prompt/target lengths. Skip this and the training section if you are only evaluating an archived run.

In [ ]:
context_id = launch('hass-context', cfg)
watch(context_id)
context = read_report(training, context_id, 'context-preflight.json')
context


## Train and read the run receipt
Confirm the measured context fits, then type `SUBMIT` to start training. A run label can be resumed but cannot be used to submit a changed plan. If submission was ambiguous, recover the printed UUID with `training.discover()` and `training.recover_submission(uuid)` before starting anything new. Private artifacts remain in the configured Slurm bucket; MLflow receives metrics and lineage.

In [ ]:
from copy import deepcopy
if context['manifest_sha256'] != pin['sha256'] or not context['fits_native_context']:
    raise ValueError('Context report does not fit this dataset and checkpoint')
train_cfg = deepcopy(cfg)
train_cfg['model']['max_seq_len'] = context['suggested_budget']
trained_id = launch('sft', train_cfg)
watch(trained_id)
summary = read_report(training, trained_id, 'training-summary.json')
print(f"MLflow: {summary['mlflow_tracking_uri'].rstrip('/')}/#/experiments/{summary['mlflow_experiment_id']}/runs/{summary['mlflow_run_id']}")
summary


## Reload and compare on DGX
Set `SOURCE_RUN_ID` in setup to evaluate a prior successful run on this exact dataset and model preset, or use the run just trained. The job verifies the archived adapter and tokenizer, reloads them, and compares two held-out examples with the adapter off/on. No Home Assistant tools are executed. Keep the report private.

In [ ]:
from copy import deepcopy
source_id = SOURCE_RUN_ID or trained_id
eval_cfg = deepcopy(cfg if SOURCE_RUN_ID else train_cfg)
eval_cfg['evaluation'] = {'source_run_id': source_id}
eval_id = launch('hass-eval', eval_cfg)
watch(eval_id)
read_report(training, eval_id, 'held-out-comparison.json')


## Restore inference
After remote jobs exit, restore the previous DGX serving profiles. This does not deploy the trained adapter.

In [ ]:
if input('Type RESTORE to return the DGX to inference: ').strip() == 'RESTORE':
    print(training.environment('inference').get('phase'))
